# 🎯 [DAT10_M03] 오프라인 단말 간 1차 피어투피어(P2P)/USB 재고 데이터 교환 Baseline 기능 구현
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT10/dat10_m03_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT10_M03 |
| **미션 주제** | 오프라인 단말 간 1차 피어투피어(P2P)/USB 재고 데이터 교환 Baseline 기능 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
전시에는 전술 통신망이 단절된 채 예하 보급소가 입고와 불출을 각자 기록하고, 통신 복구 후 기록을 병합하는 과정에서 중복·누락·충돌·위변조가 빈번히 발생하며, 이를 적시에 통제하지 못하면 재고 왜곡과 방산 공급망 위기 식별이 지연됩니다. 본 프로젝트는 방산 공급망 회복력 연구(미국 방위산업기반 정책연구)의 문제의식을 교육용 가상 자료로 재구성해, 가상 보급소 5곳이 통신 두절 환경에서 오프라인으로 재고를 기록하고 동기화하는 SaaS를 개발하고, 동기화된 데이터에서 공급망 재난 징후(업체 공급 감소, 재고 소진 임박)를 식별하는 End-to-End 군수 소프트웨어 엔지니어링 과제입니다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 03. 오프라인 단말 간 1차 피어투피어(P2P)/USB 재고 데이터 교환 Baseline 기능 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

오프라인 단말 간 피어투피어(P2P) 재고 데이터 교환 Baseline 기능을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **보급소 단말 5곳이 접촉할 때 USB·P2P로 재고 트랜잭션을 주고받는 Baseline 교환 기능을 만들고 7일 후 정보 공유 정확도 75% 이상을 확인한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **7일간 접촉 56회(접촉당 방향별 최대 80건, 손실 5%)의 가상 일정에서 상대 상태를 모르는 전체 전송, ID 차집합 델타 전송, 델타에 우선순위(오래된 것 먼저)를 더한 방식을 비교 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/models/dat10_baseline_model.py 및 기본 추론 결과 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 알고리즘의 논리적 흐름 및 군사적 판단 기준의 투명성

#### 미션 PASS 조건

- 정량 기준: 기본 데이터셋 대상 탐지/처리 정확도 75% 이상 달성


- 진행 기준: 단위 기능 동작 확인 및 기본 시나리오 실행 로그 생성


- 검증 방법: 기준 벤치마크 데이터셋 대상 추론 결과 오차율 비교


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""오프라인 보급소 5곳이 끊긴 통신 대신 짧은 P2P 접촉·USB 인편으로 트랜잭션을 교환하는 Baseline을 구현해, 전체 전송·ID 차집합(델타) 전송·델타+우선순위 휴리스틱(내 기록 우선·오래된 것 우선)의 7일 후 동기화 완전성·전송 효율·수렴 시점을 비교하고 우선순위 가중치를 1차 튜닝한다."""
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 야전 보급소·업체·품목 자료 생성기 (실제 부대·업체·품목·재고 정보 없음)
# 가상 보급소 5곳이 품목 12종의 입고·불출·조정 트랜잭션을 오프라인으로 기록하고 나중에 동기화한다.
# 공급망 재난 징후(업체 납기 지연, 품목 재고 급감)는 동기화된 자료를 마이닝해 식별한다.
# ---------------------------------------------------------------------------
NODES = ["보급소A", "보급소B", "보급소C", "보급소D", "보급소E"]
SUPPLIERS = ["가상업체1", "가상업체2", "가상업체3", "가상업체4", "가상업체5", "가상업체6"]
CATS = ["일반보급", "정비부품", "의무소모"]
ITEMS = [{"item": f"I{i + 1:02d}", "cat": CATS[i % 3], "supplier": SUPPLIERS[i % 6], "lead_days": 3 + (i * 5) % 12, "use": 4 + (i * 7) % 11, "min_level": 30 + 10 * (i % 5)} for i in range(12)]
ITEM_IDS = [x["item"] for x in ITEMS]
CTXS = ["평시", "야간", "통신불량"]


def make_txns(seed=7, days=7, per_node_day=60, nodes=None):
    """노드별 오프라인 트랜잭션 목록. id는 노드-일련번호(멱등 키), t는 시각(시간 단위), ctx는 입력 상황."""
    rng = np.random.default_rng(seed)
    out = []
    for n in (nodes or NODES):
        seq = 0
        for k in range(days * per_node_day):
            seq += 1
            it = ITEMS[int(rng.integers(len(ITEMS)))]
            t = float(rng.uniform(0, 24 * days))
            ctx = "야간" if (t % 24 >= 20 or t % 24 < 5) else "평시"
            if rng.random() < 0.12:
                ctx = "통신불량"
            r = rng.random()
            if r < 0.22:
                op, q = "in", int(it["use"] * rng.uniform(4, 9))
            elif r < 0.95:
                op, q = "out", int(max(1, rng.normal(it["use"] * 1.5, 3)))
            else:
                op, q = "adjust", int(rng.integers(-5, 6))
            out.append({"id": f"{n}-{seq:05d}", "node": n, "item": it["item"], "op": op, "qty": q, "t": round(t, 2), "ctx": ctx, "supplier": it["supplier"]})
    return out


def stock_of(txns, base=500):
    """품목별 재고 = 기초 재고 + 입고 - 불출 + 조정 (id 중복은 한 번만 반영)."""
    # [TODO : 학생 실습 3] 오프라인 단말 간 1차 피어투피어(P2P)/USB 재고 데이터 교환 Baseline 기능 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
TX = make_txns(7, 7, per_node_day=60)                          # 5노드 x 7일 x 60건 = 2,100건
N = len(TX); IDX = {x["id"]: i for i, x in enumerate(TX)}
ORIGIN = np.array([NODES.index(x["node"]) for x in TX]); T = np.array([x["t"] for x in TX])
CONTACTS_PER_DAY, BW, LOSS = 8, 80, 0.05                       # 하루 접촉 8회, 접촉당 방향별 최대 60건, 전송 손실 5%


def contact_schedule(seed=3, days=7):
    # [TODO : 학생 실습 3] 오프라인 단말 간 1차 피어투피어(P2P)/USB 재고 데이터 교환 Baseline 기능 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def simulate(method, w_own=0.0, w_age=0.0, seed=3, trace=False):
    r = np.random.default_rng(seed + 100)
    known = [set() for _ in range(5)]; sent = useful = 0; tr = []
    prod = [np.where(ORIGIN == n)[0] for n in range(5)]
    for tm, a, b, _ in contact_schedule(seed):
        for n in (a, b):                                       # 접촉 시점까지 자기 노드가 기록한 트랜잭션
            known[n] |= set(prod[n][T[prod[n]] <= tm].tolist())
        for src, dst in ((a, b), (b, a), ):
            if method == "full":
                cand = sorted(known[src], key=lambda i: T[i])    # 상대가 뭘 가졌는지 모르고 전부 보냄(앞에서부터 BW건)
            else:
                cand = [i for i in known[src] if i not in known[dst]]        # 상대가 없는 것만(ID 요약 비교)
                if method == "prio":
                    cand.sort(key=lambda i: -(w_own * (ORIGIN[i] == src) + w_age * (tm - T[i]) / 168))
                else:
                    cand.sort(key=lambda i: T[i])
            for i in cand[:BW]:
                sent += 1
                if r.random() > LOSS:
                    useful += i not in known[dst]; known[dst].add(i)
        if trace:
            tr.append((tm, np.mean([len(k) for k in known]) / N))
    comp = float(np.mean([len(k) for k in known]) / N)
    return {"comp": comp, "sent": sent, "useful": useful, "eff": useful / max(sent, 1), "full_nodes": sum(len(k) == N for k in known), "trace": tr}


# 1) 방식 비교
res = {"전체 전송(상대 상태 모름)": simulate("full"), "ID 차집합 델타(오래된 것 먼저)": simulate("delta"), "델타 + 우선순위(기본 가중 own 1, age 0)": simulate("prio", 1, 0)}
print(f"[조건] 보급소 5곳, 트랜잭션 {N}건, 7일간 접촉 {CONTACTS_PER_DAY * 7}회(P2P·USB), 접촉당 방향별 최대 {BW}건, 손실 {LOSS:.0%}")
print(f"{'방식':32s} {'완전성':>7s} {'전송':>6s} {'유효 전송':>8s} {'효율':>6s}")
for k, v in res.items():
    print(f"{k:32s} {v['comp']:7.1%} {v['sent']:6d} {v['useful']:8d} {v['eff']:6.1%}")

# 2) 1차 튜닝: 우선순위 가중치 격자 탐색(스케줄 시드 3개 평균, 시험은 다른 시드 3개)
grid = {}
for wo, wa in itertools.product((0, 1, 2, 4), (-1, 0, 1)):
    grid[(wo, wa)] = np.mean([simulate("prio", wo, wa, seed=s)["comp"] for s in (1, 2, 4)])
best = max(grid, key=grid.get)
print(f"\n[우선순위 가중치 탐색(튜닝 시드 3개 평균 완전성)] 내 기록 가중 x 오래됨 가중")
print("  " + " ".join(f"own{wo}/age{wa}:{v:.1%}" for (wo, wa), v in list(grid.items())[:6]) + " ...")
print(f"채택: own {best[0]}, age {best[1]} (튜닝 평균 {grid[best]:.1%})")
test = {m: np.mean([simulate(m, *( (best if m == 'prio' else (0, 0)) ), seed=s)["comp"] for s in (5, 6, 7)]) for m in ("full", "delta", "prio")}
print(f"시험 시드 3개 평균 완전성: 전체 전송 {test['full']:.1%}, 델타 {test['delta']:.1%}, 델타+튜닝 우선순위 {test['prio']:.1%}")
acc = test["prio"]
print(f"기본 정확도(7일 후 모든 노드가 전체 트랜잭션 중 알고 있는 비율) {acc:.1%} (목표 75% 이상: {'달성' if acc >= 0.75 else '미달'})")

# 3) 수렴 추적과 판단 지연
import time
t0 = time.perf_counter(); r = simulate("prio", *best, trace=True); dt = time.perf_counter() - t0
tr = r["trace"]; day = next((t / 24 for t, c in tr if c >= 0.9), None)
print(f"\n[수렴] 7일 시뮬레이션 {dt * 1000:.0f}ms (접촉 1회당 {dt / len(tr) * 1000:.1f}ms), 완전성 90% 도달 시점 {('%.1f일' % day) if day else '7일 내 미도달'}, 모든 트랜잭션을 가진 노드 {r['full_nodes']}/5")
print("접촉이 부족하면 완전성이 100%에 못 미치므로, 동기화 전 재고는 '확인된 범위' 표시가 필요하다(M6).")
print("※ 가상 접촉 일정과 대역폭 가정에 따른 결과이며, 실제 접촉 빈도·USB 이동은 현장 조건에 따라 다르다. 병합 충돌 판단은 M4·M7에서 다룬다.")

plt.figure(figsize=(8, 4))
for m, lab in (("full", "full dump"), ("delta", "delta"), ("prio", "delta + priority")):
    rr = simulate(m, *(best if m == "prio" else (0, 0)), trace=True)["trace"]; plt.plot([t / 24 for t, _ in rr], [c * 100 for _, c in rr], label=lab)
plt.xlabel("days"); plt.ylabel("mean completeness (%)"); plt.title("P2P/USB exchange: completeness over time"); plt.legend(); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 03 예상 실행 결과 (Output)</b></summary>

```text
[조건] 보급소 5곳, 트랜잭션 2100건, 7일간 접촉 56회(P2P·USB), 접촉당 방향별 최대 80건, 손실 5%
방식                                   완전성     전송    유효 전송     효율
전체 전송(상대 상태 모름)                    22.8%   8470      398   4.7%
ID 차집합 델타(오래된 것 먼저)                82.5%   6999     6657  95.1%
델타 + 우선순위(기본 가중 own 1, age 0)      83.7%   7143     6791  95.1%

[우선순위 가중치 탐색(튜닝 시드 3개 평균 완전성)] 내 기록 가중 x 오래됨 가중
  own0/age-1:84.8% own0/age0:83.7% own0/age1:82.7% own1/age-1:84.7% own1/age0:84.4% own1/age1:84.0% ...
채택: own 0, age -1 (튜닝 평균 84.8%)
시험 시드 3개 평균 완전성: 전체 전송 24.0%, 델타 82.1%, 델타+튜닝 우선순위 84.6%
기본 정확도(7일 후 모든 노드가 전체 트랜잭션 중 알고 있는 비율) 84.6% (목표 75% 이상: 달성)

[수렴] 7일 시뮬레이션 58ms (접촉 1회당 1.0ms), 완전성 90% 도달 시점 7일 내 미도달, 모든 트랜잭션을 가진 노드 0/5
접촉이 부족하면 완전성이 100%에 못 미치므로, 동기화 전 재고는 '확인된 범위' 표시가 필요하다(M6).
※ 가상 접촉 일정과 대역폭 가정에 따른 결과이며, 실제 접촉 빈도·USB 이동은 현장 조건에 따라 다르다. 병합 충돌 판단은 M4·M7에서 다룬다.
```
</details>
